# 21 - Does the engine lie? PixInsight's arithmetic, against known answers

**Purpose.** Establish that the two operations build step 5 rests on - **subtracting one frame
from another**, and **integrating a stack** - return what was put into them, and publish what
each wrong setting costs. Everything here is measured on **synthetic frames with an injected
sigma**, because the whole point is to compare against an answer known in advance.

This is the gate `LEGACY` L23 put at the head of the work: *the first thing it must verify,
before any number it produces is believed*. It is not contract 2 and not contract 3 - it is
what makes both of them readable. Contract 2 measures `g` and `R` on a real bias pair through
PixInsight, and it subtracts; contract 3 measures `eta_comb` on real frames, and it integrates.
Neither can be believed until the arithmetic under it has been checked, and neither can check it,
because on real frames every wrong answer here looks plausible.

**What it is not for.** No real frame is opened. No published constant is measured, re-measured
or corrected: `g`, `R`, the pedestal and session 03's `eta_comb` are all untouched, and nothing
in `results/` moves because of this notebook except its own two files. Not registration - that is
contract 3's, and every stack here is deliberately unregistered. Not a comparison of estimators:
`19` did that, and this notebook borrows its conclusion rather than repeating it.

**Why synthetic, stated before the first cell.** L23 was explicit that its claim must not be
checked by inspecting a real bias pair, and the reason generalises to everything below. A
subtraction that clips reads **42% low**, and 42% low on a real bias looks like a better camera.
A stack that loses 3% to its rejection settings looks like a stack. There is no feature of a real
frame that reveals either. An injected sigma reveals both immediately, and it is the only thing
that does.

**It assumes `00_statistics.ipynb`** for why a spread over a quarter of a million pixels is a
sharp number, and `20_pi_contract_read.ipynb` for why PixInsight's MRS rather than our
`1.4826 x MAD` is the right referee wherever a spread is a few counts.

**Four things this notebook found that were not in any claim**, named here rather than left to be
discovered in section 6: `ImageIntegration` refuses fewer than three source images; its default
weighting refuses a frame without stars; `for...in` over a process prototype takes the core down;
and PixInsight's console *is* retrievable from inside a script, which is how the second of those
was diagnosed at all. All four are in `pjsr/NOTES.md`.

In [ ]:
import json
import pathlib
import shutil
import sys
import tempfile

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, pixinsight as PI, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"

# One PI value is this many ADC counts (L20, and `pixinsight.to_adc`).  Spelled
# out of the module rather than written as 4095.9375, so there is one source.
SCALE = stats.STORED_FULL_SCALE / (1 << stats.ADC_SHIFT)
STEP = 1 << stats.ADC_SHIFT             # stored = ADC count << 4

# The synthetic frames.  512x512 is a quarter of a million pixels per frame --
# enough that a sigma is sharp to a fraction of a percent, small enough that
# thirty-two of them and a per-pixel census cost seconds rather than minutes.
SHAPE = (512, 512)
PEDESTAL_ADC = 77.0                     # counts; a plausible black level
SIGMA_NARROW = 20.0                     # counts per frame
SIGMA_WIDE = 600.0                      # counts per frame, to find the float's limit
N_STACK = 32
RUNGS = [3, 4, 8, 16, 32]               # 2 is unavailable -- see section 5
SEED = 20260919

ARMS_CSV = RESULTS / "pi_arithmetic_arms.csv"
LADDER_CSV = RESULTS / "pi_arithmetic_ladder.csv"
CONSTANTS = RESULTS / "pi_arithmetic_constants.json"

# Intermediates are disposable and C: is tight (CLAUDE.md): the frames live in a
# temp directory and the last cell removes it.
WORK = pathlib.Path(tempfile.mkdtemp(prefix="astropix-21-"))
print(f"synthetic frames in {WORK}")

probe = PI.run(PI.scripts_dir() / "probe.js", {"an_int": 42})
assert probe["ok"], probe.get("error")
core = probe["core"]
print(f"PixInsight {core['version']} build {core['build']}, instance slot {core['instance']}")

## 1. The frames, and the truth they carry

Gaussian noise about a pedestal, **quantised onto the ADC grid and stored the way the camera
stores it** - values are exact multiples of 16 in the file, so `stats.to_adc` has the same work to
do here as on a real frame and the `% 16` check that licenses the unit convention still means
something. A synthetic frame that skipped that would be testing a code path the camera never uses.

The pedestal is not decoration. A difference of two frames is centred on zero *whatever* the
pedestal is, because it cancels - which is precisely why the trap below is about the difference
and not about the frames.

`truth` is the numpy sigma of the difference, computed on the same integers PixInsight will read.
Not `sigma * sqrt(2)`: quantisation moves it by a hair, and the comparison below is between two
tools reading one file, not between a tool and an ideal.

In [ ]:
rng = np.random.default_rng(SEED)


def write_frame(path, sigma, pedestal=PEDESTAL_ADC):
    # One synthetic frame, in stored units, on the ADC grid.
    v = rng.normal(pedestal, sigma, SHAPE)
    stored = np.clip(np.rint(v), 0, stats.ADC_FULL_SCALE).astype(np.uint16) * STEP
    F.write(path, stored, overwrite=True)
    return path


def read_adc(path):
    return stats.to_adc(F.read(path)[0]).astype(np.float64)


pairs = {}
for tag, sigma in (("narrow", SIGMA_NARROW), ("wide", SIGMA_WIDE)):
    ped = PEDESTAL_ADC if tag == "narrow" else 2048.0   # room for 600 counts of sigma
    a = write_frame(WORK / f"{tag}_a.fit", sigma, ped)
    b = write_frame(WORK / f"{tag}_b.fit", sigma, ped)
    truth = float((read_adc(a) - read_adc(b)).std(ddof=1))
    pairs[tag] = {"a": a, "b": b, "sigma": sigma, "truth": truth,
                  "frac_of_range": truth / SCALE}
    print(f"{tag:7s} sigma {sigma:6.1f} counts/frame -> difference {truth:9.4f} counts "
          f"= {truth / SCALE:.4f} of PI's [0, 1] range")

# The quantisation check: our own sigma_from_pair should return the injected
# value.  If this fails, nothing below is about PixInsight.
for tag, p in pairs.items():
    got = PI.sigma_from_pair(p["truth"])
    print(f"  {tag:7s} sigma_from_pair recovers {got:8.4f} vs {p['sigma']:6.1f} injected "
          f"({100 * (got / p['sigma'] - 1):+.3f}%)")
    assert abs(got / p["sigma"] - 1) < 0.01

## 2. Eight settings, one launch, and the trap in two of them

`PixelMath` is asked for `a - b + pedestal` under every combination of three choices: the pedestal
present or absent, the output 32-bit float or 16-bit integer, truncation on or off. The script
chooses none of them - they arrive in the job - **because a referee that picked the safe settings
itself could never demonstrate the unsafe ones**, and demonstrating them is the point.

All eight run in one launch. Core startup is about forty seconds and the subtraction is
milliseconds, so the alternative was five minutes of PixInsight starting up to answer one question
eight times.

Two columns carry the argument. `sigma` against the numpy truth says *what it cost*. The census -
how many pixels sit **below** zero against how many sit **exactly on** zero - says *what happened*,
and those are the same pixels meaning opposite things: kept, or clipped.

In [ ]:
ARMS = [{"label": f"pedestal {ped}, {fmt}, truncate {trunc}",
         "pedestal": ped, "sample_format": fmt, "truncate": trunc}
        for ped in (PI.DIFF_PEDESTAL, 0.0)
        for fmt in ("f32", "i16")
        for trunc in (False, True)]

rows = []
for tag, p in pairs.items():
    r = PI.run(PI.scripts_dir() / "pair_diff.js",
               {"a": PI.pi_path(p["a"]), "b": PI.pi_path(p["b"]), "arms": ARMS},
               timeout=900)
    assert r["ok"], r.get("error", "") + "\n" + str(r.get("log", ""))[-1500:]
    for arm in r["arms"]:
        d, c, s = arm["diff"], arm["census"], arm["settings"]
        rows.append({"pair": tag, "injected_sigma": p["sigma"], "truth": p["truth"],
                     "pedestal": s["pedestal"], "format": s["sample_format"],
                     "truncated": s["truncate"], "bits": d["bits_per_sample"],
                     "is_real": d["is_real"], "sigma": d["std"] * SCALE,
                     "min": d["min"], "median": d["median"],
                     "frac_below_zero": c["frac_below_zero"],
                     "frac_at_zero": c["frac_at_zero"]})

arms = pd.DataFrame(rows)
# JSON booleans arrive as objects, and an object column will not take `&`.
arms["truncated"] = arms["truncated"].astype(bool)
arms["is_real"] = arms["is_real"].astype(bool)
arms["err_pct"] = 100 * (arms.sigma / arms.truth - 1)
arms["recovered_sigma"] = arms.sigma / np.sqrt(2.0)

show = arms[arms.pair == "narrow"][
    ["pedestal", "format", "truncated", "bits", "sigma", "err_pct",
     "frac_below_zero", "frac_at_zero"]]
print(f"narrow pair -- truth {pairs['narrow']['truth']:.4f} ADC counts\n")
print(show.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### What the table says, and it is not what L23 said

**The pedestal is the whole fix, and the 32-bit float is doing nothing here.** With the pedestal
present, all four combinations of format and truncation return the truth exactly. Every failure in
the table is a difference that was left centred on zero.

That is a real departure from the inherited claim, which prescribed the float and the pedestal
together as one remedy. At any spread a bias pair can have, one of the two is load-bearing and the
other is not.

**Two arms fail quietly and one fails loudly, and the loud one is the safer failure.** Truncation
with no pedestal clips the negative half onto exactly zero and reads low. Sixteen-bit *without*
truncation does something else entirely - a negative in an unsigned container wraps - and the
result is unmistakable garbage. A number that is wrong by seventy times gets noticed; a number
that is wrong by 42% gets published.

**And the quiet failure is the default.** A fresh `PixelMath` has `truncate = true` and a sample
format of `SameAsTarget`, so the obvious way to subtract two 16-bit frames is exactly the
configuration that loses 42%, with nothing anywhere to warn you.

In [ ]:
narrow = arms[arms.pair == "narrow"]
with_ped = narrow[narrow.pedestal == PI.DIFF_PEDESTAL]
without = narrow[narrow.pedestal == 0.0]

print(f"with the pedestal, worst error across all four settings: "
      f"{with_ped.err_pct.abs().max():.4f}%")
assert with_ped.err_pct.abs().max() < 0.01, (
    "the pedestal did not save every setting; the claim in the markdown above is wrong")

print("\nwithout it:")
for r in without.itertuples():
    kind = ("clipped -- reads low and plausibly" if r.frac_at_zero > 0.4 else
            "wrapped -- reads absurdly high" if r.err_pct > 100 else
            "intact -- float keeps the negatives")
    print(f"  {r.format:4s} truncate {str(r.truncated):5s}  {r.err_pct:+10.2f}%   {kind}")

quiet = without[(without.err_pct < -10) & (without.err_pct > -90)]
loud = without[without.err_pct > 100]
print(f"\n{len(quiet)} of {len(without)} arms fail quietly, {len(loud)} loudly, "
      f"{len(without) - len(quiet) - len(loud)} not at all")

## 3. Where the float does earn its place

The pedestal buys half of PixInsight's range on each side of zero. A difference wider than that
runs off the bottom anyway, and then the format decides whether the overflow is held or lost.

The wide pair is sigma 600 counts per frame - **thirty times a real bias pair**, and chosen for
exactly that reason. It is not a spread this camera can produce; it is the spread at which the
pedestal stops being sufficient, which is the only way to find out whether the float was ever
doing anything.

In [ ]:
wide = arms[(arms.pair == "wide") & (arms.pedestal == PI.DIFF_PEDESTAL)]
print(f"wide pair -- truth {pairs['wide']['truth']:.4f} counts, "
      f"{pairs['wide']['frac_of_range']:.3f} of PI's range; "
      f"3 sigma is {3 * pairs['wide']['frac_of_range']:.3f}, against a pedestal of "
      f"{PI.DIFF_PEDESTAL}\n")
print(wide[["format", "truncated", "bits", "sigma", "err_pct",
            "frac_below_zero", "frac_at_zero", "min"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

best = wide.loc[wide.err_pct.abs().idxmin()]
print(f"\nonly {best.format} with truncate={best.truncated} is exact here "
      f"({best.err_pct:+.4f}%); the rest lose "
      f"{wide[wide.format != 'f32'].err_pct.abs().min():.2f}% to "
      f"{wide.err_pct.abs().max():.2f}%")
print("\nso the prescribed configuration is kept in full: the pedestal for every difference a "
      "bias\npair can produce, and the float for the case the pedestal cannot cover.")

## 4. The 42% is the half-normal, and it was predictable

Clipping a zero-centred normal at zero does not merely remove half the pixels - it piles them onto
a single value. What survives is a half-normal, whose second moment is half the original variance
and whose mean is no longer zero, so its standard deviation is

```
sqrt(1/2 - 1/(2*pi))  =  0.5838
```

of the truth. L23 said "halves". 0.584 is what "halves" turns out to mean, and the difference
between the two is the difference between a remembered number and a derived one.

This matters beyond bookkeeping. **The factor is a constant**, independent of the sigma and of the
frame - so a clipped difference cannot be recognised by its value at all. It is always 58% of
whatever the right answer was, which is why the census in section 2 is the evidence and the sigma
is only the cost.

In [ ]:
predicted = float(np.sqrt(0.5 - 1.0 / (2.0 * np.pi)))
clipped_arms = arms[(arms.pedestal == 0.0) & (arms["truncated"]) & (arms.frac_at_zero > 0.4)]

print(f"predicted from the half-normal: {predicted:.6f}\n")
print(f"  {'pair':8s} {'format':7s} {'measured ratio':>15s} {'vs predicted':>14s}")
for r in clipped_arms.itertuples():
    ratio = r.sigma / r.truth
    print(f"  {r.pair:8s} {r.format:7s} {ratio:15.6f} {100 * (ratio / predicted - 1):+13.3f}%")

spread = (clipped_arms.sigma / clipped_arms.truth)
assert abs(spread.mean() / predicted - 1) < 0.01, (
    "the clipped arms do not land on the half-normal; the mechanism is not what section 4 says")
print(f"\nthe factor does not depend on the sigma: the narrow pair and the wide pair are "
      f"{pairs['wide']['sigma'] / pairs['narrow']['sigma']:.0f}x apart in spread and land "
      f"{100 * (spread.max() / spread.min() - 1):.2f}% apart in ratio")

## 5. The stack, and the two refusals on the way to it

`ImageIntegration` on frames whose noise is known, at five stack sizes, with rejection off and
then on. With rejection off and equal weights, averaging `N` independent frames should divide the
noise by exactly `sqrt(N)`; anything less is the engine losing something, and that is worth
knowing before a real `eta_comb` is ever quoted.

**Two settings had to be found the hard way, and neither announces itself.**

*Three source images is a hard floor.* Fewer and `executeGlobal` refuses by name. So the **N=2
rung of a doubling ladder cannot be measured through this engine at all** - which matters, because
session 03's `eta_comb` ladder on darks has one. The ladder here starts at 3.

*The default weighting refuses a frame without stars.* `weightMode` defaults to **PSF Signal
Weight**, which weights each frame by its detected stars; a synthetic frame has none, the weight
comes out zero, and `executeGlobal()` returns a bare `false` having written "Zero or insignificant
PSF Signal Weight estimate" to a console the caller cannot see. It cost three launches and was
diagnosed only once `harness.jsh` began capturing the core's own log. So the weighting is stated
here and never inherited - and `dont_care` is the right choice regardless, because `eta_comb`
against the ideal `sqrt(N)` is *defined* on equally weighted frames.

In [ ]:
stack = [write_frame(WORK / f"stack_{i:02d}.fit", SIGMA_NARROW) for i in range(N_STACK)]
sigma_single = float(read_adc(stack[0]).std(ddof=1))
print(f"{N_STACK} frames, injected {SIGMA_NARROW} counts; frame 0 measures {sigma_single:.4f}\n")

runs = [{"label": f"N={n}, rejection {rej}",
         "frames": [PI.pi_path(p) for p in stack[:n]],
         "combination": "average", "rejection": rej, "normalization": "none",
         "weight_mode": "dont_care", "sigma_low": 4.0, "sigma_high": 3.0}
        for rej in ("none", "winsorized") for n in RUNGS]

r = PI.run(PI.scripts_dir() / "integrate.js", {"runs": runs}, timeout=1800)
assert r["ok"], r.get("error", "") + "\n" + str(r.get("log", ""))[-1500:]

rows = []
for run in r["runs"]:
    assert run["ok"], f"{run['label']}: {run.get('error')}"
    s = run["settings"]
    rows.append({"n": run["n"], "rejection": s["requested"]["rejection"],
                 "sigma_stack": run["integrated"]["std"] * SCALE,
                 "mrs": (run["noise"]["mrs"] or np.nan) * SCALE,
                 "bits": run["integrated"]["bits_per_sample"],
                 "weight_mode": s["weight_mode"], "sigma_low": s["sigma_low"],
                 "sigma_high": s["sigma_high"], "normalization": s["normalization"]})

ladder = pd.DataFrame(rows)
ladder["sigma_single"] = sigma_single
ladder["ideal"] = sigma_single / np.sqrt(ladder.n)
ladder["eta_comb"] = [PI.eta_comb(sigma_single, r.sigma_stack, r.n)
                      for r in ladder.itertuples()]
print(ladder[["n", "rejection", "sigma_stack", "ideal", "eta_comb", "mrs"]]
      .to_string(index=False, float_format=lambda v: f"{v:.5f}"))

### Averaging loses nothing, so `eta_comb` is a number about the rejection

With rejection off the engine reaches the ideal at every rung to within a fraction of a percent -
scatter on the estimate rather than a loss. **That settles what `eta_comb` is measuring.** It is
not the arithmetic, which is exact. It is the *combination*: the rejection algorithm and its
thresholds, the normalisation, the weighting, and - once frames are registered - the resampling
kernel.

With Winsorized clipping at 4.0/3.0 it costs several percent, and **the cost is a strong function
of the stack size**, worst at N=3 and shrinking as the stack grows. Rejection discards real pixels,
and at three frames there are too few survivors left to average well.

That is the measurement behind MISSION's requirement that `eta_comb`'s provenance record the stack
size and the rejection settings. The same setting costs several times more at one end of the
ladder than the other, so a number quoted without both is not a measurement of anything.

**These frames have no outliers.** No cosmic rays, no satellites, no registration, no structure -
so the rejection column is the **pure cost of rejecting when there is nothing to reject**, which
is a floor and not an estimate. On real frames rejection buys something back, and what it buys is
contract 3's to measure, not this notebook's.

In [ ]:
clean = ladder[ladder.rejection == "none"]
rejected = ladder[ladder.rejection == "winsorized"]

worst = (clean.eta_comb - 1.0).abs().max()
print(f"rejection off: eta_comb within {100 * worst:.2f}% of 1.0 at every rung "
      f"({clean.n.min()} to {clean.n.max()})")
assert worst < 0.01, "the engine is not reaching sqrt(N); section 5's claim is wrong"

print(f"\nrejection on, cost against the ideal:")
for r in rejected.sort_values("n").itertuples():
    print(f"  N={r.n:3d}  eta_comb {r.eta_comb:.4f}   costs {100 * (1 - r.eta_comb):5.2f}%")
print(f"\nthe cost falls {100 * (1 - rejected.eta_comb.min()):.2f}% -> "
      f"{100 * (1 - rejected.eta_comb.max()):.2f}% from N={rejected.loc[rejected.eta_comb.idxmin()].n:.0f} "
      f"to N={rejected.loc[rejected.eta_comb.idxmax()].n:.0f}, which is why the stack size is "
      f"part of the constant and not context for it")

## 6. What this licenses, and the two files it writes

The tables go to `results/` with the build that produced them, because a number measured against
an unnamed version of a referee is not a number. Everything here is a property of **PixInsight
1.9.2 and this project's settings**, not of the sensor - no constant in MISSION's table is touched
by any of it.

In [ ]:
arms.to_csv(ARMS_CSV, index=False)
ladder.to_csv(LADDER_CSV, index=False)
print(f"wrote {ARMS_CSV.name} ({len(arms)} rows) and {LADDER_CSV.name} ({len(ladder)} rows)")

measured_on = "2026-09-19"


def constant(value, unit, uncertainty, source_frames, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": source_frames, "measured_on": measured_on,
            "notebook": "21_pi_arithmetic.ipynb", "note": note}


def nn(v):
    return None if v is None or not np.isfinite(v) else round(float(v), 9)


n_pair_frames = 2 * len(pairs)
measured_clip = float((clipped_arms.sigma / clipped_arms.truth).mean())

constants = {
    "referee": constant(
        {"application": "PixInsight", "version": core["version"], "build": core["build"]},
        None, None, 0,
        "the build every number in this file was measured against.  A different build is a "
        "different engine; re-run pjsr/probe.js after any upgrade.  See pjsr/NOTES.md"),
    "difference_settings": constant(
        {"pedestal": PI.DIFF_PEDESTAL, "sample_format": "f32", "truncate": False,
         "rescale": False, "expression": "a - b + 0.5"},
        None, None, n_pair_frames,
        "the PixelMath configuration every pair difference in this project must use.  The "
        "PEDESTAL is what matters at any spread a bias pair can have -- with it, all four "
        "combinations of format and truncation recover the injected sigma exactly.  The 32-bit "
        "float covers only the case where the difference is wide enough to leave [0, 1] anyway, "
        "which no bias pair is.  Both are kept because the second costs nothing.  NOTE that "
        "truncate defaults to TRUE and the format to SameAsTarget, so the default configuration "
        "is the failing one"),
    "clip_factor": constant(
        nn(measured_clip), "fraction of the true sigma", nn(
            float((clipped_arms.sigma / clipped_arms.truth).std(ddof=1))
            if len(clipped_arms) > 1 else 0.0),
        n_pair_frames,
        f"what a difference clipped at zero reports, as a fraction of the truth.  Predicted "
        f"sqrt(1/2 - 1/(2*pi)) = {predicted:.6f} from the half-normal and measured here on two "
        "pairs whose spreads differ by a factor of thirty.  L23 called it 'halves'.  It is a "
        "CONSTANT, independent of the sigma and of the frame, so a clipped difference cannot be "
        "recognised from its value -- which is why the pixel census rather than the sigma is the "
        "evidence that clipping happened"),
    "wrap_on_unsigned": constant(
        nn(float(arms[(arms.pedestal == 0.0) & (arms.format == "i16")
                      & (~arms["truncated"]) & (arms.pair == "narrow")].err_pct.iloc[0])),
        "percent error", None, n_pair_frames,
        "a negative difference stored in an unsigned 16-bit container with truncation OFF wraps "
        "rather than clips.  The failure mode L23 did not name, and the safe one: an error this "
        "size cannot be mistaken for a measurement, where the 42% of the clip can"),
    "integration_minimum_frames": constant(
        3, "frames", 0, 0,
        "ImageIntegration refuses executeGlobal with fewer -- 'This instance of ImageIntegration "
        "defines less than three source images' -- whatever the rejection setting.  So the N=2 "
        "rung of a doubling ladder cannot be measured through this engine at all, which matters "
        "because session 03's eta_comb ladder on darks has one"),
    "integration_weight_mode": constant(
        "dont_care", None, None, N_STACK,
        "weightMode MUST be stated.  It defaults to PSF Signal Weight, which weights each frame "
        "by its detected stars, and on a frame without any -- a bias, a dark, a flat, a synthetic "
        "frame -- the weight is zero and executeGlobal returns a bare false with its reason only "
        "in the console.  dont_care is also the right choice on merit: eta_comb against the ideal "
        "sqrt(N) is defined on equally weighted frames"),
    "eta_comb_engine": constant(
        {int(r.n): nn(r.eta_comb) for r in clean.itertuples()},
        "dimensionless", nn(float(worst)), N_STACK,
        "combination efficiency of the ENGINE ALONE: average combination, no rejection, equal "
        "weights, no normalisation, unregistered synthetic frames with no structure.  It is 1.0 "
        "to within a fraction of a percent at every rung, which is the result -- averaging loses "
        "nothing, so any shortfall in a real eta_comb is the combination and never the "
        "arithmetic.  NOT a value to use as eta_comb anywhere: it is a calibration of the tool"),
    "eta_comb_rejection_cost": constant(
        {int(r.n): nn(r.eta_comb) for r in rejected.itertuples()},
        "dimensionless", None, N_STACK,
        "the same ladder with Winsorized sigma clipping at sigma_low=4.0, sigma_high=3.0.  The "
        "cost is a strong function of the stack size -- worst at N=3, shrinking as N grows -- "
        "which is why MISSION requires eta_comb's provenance to record both.  These frames "
        "contain NO outliers, so this is the pure cost of rejecting when there is nothing to "
        "reject: a floor, not an estimate.  On real frames rejection buys something back, and "
        "that is contract 3's measurement"),
    "synthetic_frames": constant(
        {"shape": list(SHAPE), "pedestal_adc": PEDESTAL_ADC, "seed": SEED,
         "sigma_narrow_adc": SIGMA_NARROW, "sigma_wide_adc": SIGMA_WIDE,
         "n_stack": N_STACK, "rungs": RUNGS},
        None, None, n_pair_frames + N_STACK,
        "how the frames were made.  Gaussian noise about a pedestal, quantised onto the ADC grid "
        "and written in STORED units so that the reader has the same work to do as on a real "
        "frame.  Synthetic on purpose: L23 forbade checking its claim on a real bias pair, "
        "because a subtraction that reads 42% low there looks like a better camera"),
}

with open(CONSTANTS, "w", encoding="utf-8") as fh:
    json.dump(constants, fh, indent=2)
print(f"wrote {CONSTANTS.name} with {len(constants)} constants")
for k, v in constants.items():
    print(f"  {k:30s} {str(v['value'])[:70]}")

### What is licensed, and what is not

**Licensed.** A pair difference taken through PixInsight under the published settings returns the
truth, and the cost of every wrong setting is on record. A stack integrated through
`ImageIntegration` with rejection off and equal weights loses nothing to the engine. Both can now
be used to measure something about the sensor, which neither could before.

**Not licensed.** `eta_comb_engine` is **not** `eta_comb`. It is a calibration of the tool on
frames with no structure, no outliers and no registration, and using it as the model's combination
efficiency would be quoting the instrument's own reading as the measurement. The real number needs
real frames, and the registered one needs contract 3.

**Not touched.** No published constant moved. `g`, `R`, the pedestal, `F_sky`, `t_dead` and
session 03's `eta_comb` are exactly where they were; this notebook measured the engine, not the
camera.

**One launch is one build.** Every number here is a property of PixInsight 1.9.2 build 1632. The
half-normal factor in section 4 is the exception and belongs to arithmetic rather than to any
version of anything - which is why it is derived in the markdown and asserted in
`tests/test_pixinsight.py` in pure numpy, where it survives having no PixInsight at all.

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
print(f"removed {WORK}")
print(f"  {len(pairs) * 2 + N_STACK} synthetic frames, "
      f"{(len(pairs) * 2 + N_STACK) * np.prod(SHAPE) * 2 / 1e6:.0f} MB, gone -- "
      f"they are reproducible from the seed in synthetic_frames and C: is tight")